# Evaluation: golden questions, retrieval, the agent and DeepEval

Measuring whether answers are right, and why they are not.

See the [README](README.md) for the why and the how.

In [ ]:
# Run from anywhere inside the repository: find its root so `marginalia` and the test helpers import,
# and read .env (the same settings the services use).
import os, sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(ROOT)

In [ ]:
from marginalia.evals.golden import load_golden
gold = load_golden('evals/golden/crime-and-punishment.jsonl')
import collections
print(len(gold), 'questions', collections.Counter(q.category for q in gold))
gold[0]

## A saved agent run

In [ ]:
import json
from pathlib import Path
run = Path('evals/results/baseline-qwen3-4b')
summary = json.loads((run / 'summary.json').read_text())
print({k: summary[k] for k in ('accuracy', 'accuracy_ci95') if k in summary})
print((run / 'report.md').read_text()[:1500])

## Retrieval only: is a gold chapter in the top k? (needs an indexed Crime and Punishment)

In [ ]:
from marginalia.evals import runner
from marginalia.index.retriever import Retriever
from marginalia.library import Library
try:
    profile = Library().resolve('crime')
    rows = runner.run_retrieval(gold, Retriever.for_book(profile), k=10)
    print(runner.summarize_retrieval(rows, k=10))
except Exception as e:
    print('skipped:', type(e).__name__, e)

## DeepEval metrics, judged by the local model
The judge asks Ollama in plain text and reads the JSON from the reply (qwen3:4b returns nothing when asked for a schema while thinking).

In [ ]:
from marginalia.evals import deepeval_suite as de
print(de.last_json('Let me think ... {"verdicts": [{"verdict": "yes"}]}'))
record = {'answer': 'He killed Alyona Ivanovna and Lizaveta [Part I, Chapter VII].',
          'evidence': [{'citation': 'Part I, Chapter VII, passage 3', 'text': 'He struck the old pawnbroker with the axe; then Lizaveta came in and he killed her too.'}]}
case = de.test_case(gold[0], record)
case

In [ ]:
# Needs Ollama: one metric, one question (a few minutes on a small GPU).
judge = de.make_judge()
metric = de.make_metrics(judge, ['faithfulness'])['faithfulness']()
await metric.a_measure(case, _show_indicator=False)
metric.score, metric.reason

## From the command line
```bash
python -m marginalia.evals retrieval --book crime
python -m marginalia.evals agent --book crime --runs 1
python -m marginalia.evals deepeval --book crime --metrics faithfulness,correctness
```